# WEEK 4 — Major Project (Capstone Project)
## Google Colab — Solved Projects

This notebook implements all three project options listed in the Week 4 assignment:
1. Heart Disease Prediction
2. Bank Loan Approval Prediction
3. Customer Segmentation using Clustering

The assignment specifies a full ML pipeline with documentation and presentation. The dataset names are Kaggle datasets, so each section includes a Colab upload step for the corresponding CSV.


In [ ]:
# Install/import the required libraries
!pip -q install xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_curve, auc,
    silhouette_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")


# Option 1 — Heart Disease Prediction

**Dataset:** Kaggle — Heart Disease UCI

**Required workflow:** data cleaning + feature scaling → Logistic Regression / Decision Tree → accuracy, precision, recall → confusion matrix + ROC curve.

**Outcome:** predict whether a patient is likely to have heart disease.


In [ ]:
# Upload your Heart Disease CSV in Google Colab
uploaded = files.upload()
heart_file = next(iter(uploaded))
heart = pd.read_csv(heart_file)

print("Shape:", heart.shape)
display(heart.head())
print("\nColumns:", heart.columns.tolist())
print("\nMissing values:")
display(heart.isnull().sum())


In [ ]:
# Identify the target column.
# Common names in Heart Disease UCI datasets include: target, condition, num.
possible_targets = ["target", "condition", "num", "output", "HeartDisease", "heartdisease"]

target_col = next((c for c in possible_targets if c in heart.columns), None)

if target_col is None:
    print("Target column not detected automatically.")
    print("Set target_col manually, for example: target_col = 'target'")
else:
    print("Detected target:", target_col)

# If needed, uncomment and edit:
# target_col = "target"

assert target_col is not None, "Please set target_col to the target column in your dataset."

# Convert target to binary where possible.
y_raw = heart[target_col]
if y_raw.dtype == "object":
    y = pd.Series(LabelEncoder().fit_transform(y_raw.astype(str)), index=heart.index)
else:
    y = pd.to_numeric(y_raw, errors="coerce")

# Drop rows with missing target
valid = y.notna()
heart = heart.loc[valid].copy()
y = y.loc[valid].astype(int)

X = heart.drop(columns=[target_col])

# Convert categorical columns to numeric using one-hot encoding.
X = pd.get_dummies(X, drop_first=True)

# Replace inf values and impute missing values.
X = X.replace([np.inf, -np.inf], np.nan)
X = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(X), columns=X.columns)

print("Final feature matrix:", X.shape)


In [ ]:
# Train/test split + feature scaling
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Logistic Regression
log_reg = LogisticRegression(max_iter=2000)
log_reg.fit(X_train_scaled, y_train)

# Train Decision Tree
tree = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)
tree.fit(X_train, y_train)

# Predictions
lr_pred = log_reg.predict(X_test_scaled)
tree_pred = tree.predict(X_test)

def evaluate_classifier(name, y_true, y_pred):
    print(f"\n===== {name} =====")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, zero_division=0))
    print("Recall   :", recall_score(y_true, y_pred, zero_division=0))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

evaluate_classifier("Logistic Regression", y_test, lr_pred)
evaluate_classifier("Decision Tree", y_test, tree_pred)


In [ ]:
# Confusion matrix + ROC curve for Logistic Regression
cm = confusion_matrix(y_test, lr_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Heart Disease — Logistic Regression Confusion Matrix")
plt.show()

# ROC curve requires probability estimates
lr_prob = log_reg.predict_proba(X_test_scaled)[:, 1]
fpr, tpr, _ = roc_curve(y_test, lr_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Heart Disease — ROC Curve")
plt.legend()
plt.show()

print("ROC-AUC:", roc_auc)


In [ ]:
# Example: predict a new patient after preparing the same features.
# IMPORTANT: use the exact same feature columns/order as X_train.
#
# new_patient = pd.DataFrame([{
#     "age": 55,
#     "sex": 1,
#     "cp": 1,
#     "trestbps": 130,
#     "chol": 240,
#     "fbs": 0,
#     "restecg": 1,
#     "thalach": 150,
#     "exang": 0,
#     "oldpeak": 1.2,
#     "slope": 1,
#     "ca": 0,
#     "thal": 2
# }])
#
# new_patient = pd.get_dummies(new_patient, drop_first=True)
# new_patient = new_patient.reindex(columns=X.columns, fill_value=0)
# new_patient_scaled = scaler.transform(new_patient)
# prediction = log_reg.predict(new_patient_scaled)[0]
# print("Predicted class:", prediction)


# Option 2 — Bank Loan Approval Prediction

**Dataset:** Kaggle — Loan Prediction Dataset

**Required workflow:** clean categorical data such as Gender, Marital Status and Income → Random Forest / XGBoost → confusion matrix → loan approval prediction.

**Outcome:** practical classification problem.


In [ ]:
# Upload the Loan Prediction CSV
uploaded = files.upload()
loan_file = next(iter(uploaded))
loan = pd.read_csv(loan_file)

print("Shape:", loan.shape)
display(loan.head())
print("\nColumns:", loan.columns.tolist())
display(loan.isnull().sum())


In [ ]:
# Detect the target column used by common Loan Prediction datasets.
possible_targets = [
    "Loan_Status", "loan_status", "LoanStatus",
    "approved", "Approved", "target"
]
loan_target = next((c for c in possible_targets if c in loan.columns), None)

if loan_target is None:
    print("Target not detected. Set loan_target manually, e.g. loan_target = 'Loan_Status'.")
else:
    print("Detected target:", loan_target)

assert loan_target is not None, "Please set loan_target manually."

# Separate target and features
y = loan[loan_target].copy()

# Convert common Y/N target to 1/0
if y.dtype == "object":
    y = y.astype(str).str.strip().str.upper().map({
        "Y": 1, "YES": 1, "APPROVED": 1, "1": 1,
        "N": 0, "NO": 0, "REJECTED": 0, "0": 0
    })

if y.isna().any():
    # Fallback for arbitrary categorical labels
    y = pd.Series(LabelEncoder().fit_transform(loan[loan_target].astype(str)),
                  index=loan.index)

X = loan.drop(columns=[loan_target])

# Remove identifier columns if present
for c in ["Loan_ID", "loan_id", "ID", "id"]:
    if c in X.columns:
        X = X.drop(columns=[c])

# Separate numeric/categorical columns
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(exclude=np.number).columns.tolist()

# Preprocessing
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", __import__("sklearn").preprocessing.OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)

print("Processed training shape:", X_train_p.shape)


In [ ]:
# Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)
rf.fit(X_train_p, y_train)

rf_pred = rf.predict(X_test_p)

print("===== Random Forest =====")
print("Accuracy :", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred, zero_division=0))
print("Recall   :", recall_score(y_test, rf_pred, zero_division=0))
print("\nClassification Report:")
print(classification_report(y_test, rf_pred, zero_division=0))

cm = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Loan Approval — Random Forest Confusion Matrix")
plt.show()


In [ ]:
# Optional XGBoost model
xgb = XGBClassifier(
    n_estimators=250,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train_p, y_train)
xgb_pred = xgb.predict(X_test_p)

print("===== XGBoost =====")
print("Accuracy :", accuracy_score(y_test, xgb_pred))
print("Precision:", precision_score(y_test, xgb_pred, zero_division=0))
print("Recall   :", recall_score(y_test, xgb_pred, zero_division=0))
print("\nClassification Report:")
print(classification_report(y_test, xgb_pred, zero_division=0))


# Option 3 — Customer Segmentation using Clustering

**Dataset:** Kaggle — Mall Customers Dataset

**Required workflow:** apply K-Means → visualize groups using Age / Income / Spending Score → label groups as Low Spenders, Medium Spenders and High Spenders.

**Outcome:** business analytics-oriented project.


In [ ]:
# Upload the Mall Customers CSV
uploaded = files.upload()
mall_file = next(iter(uploaded))
mall = pd.read_csv(mall_file)

print("Shape:", mall.shape)
display(mall.head())
print("Columns:", mall.columns.tolist())


In [ ]:
# Detect the required columns.
def find_col(df, candidates):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    # case-insensitive fallback
    lower = {c.lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    return None

age_col = find_col(mall, ["Age"])
income_col = find_col(mall, ["Annual Income (k$)", "Annual Income", "Income"])
score_col = find_col(mall, ["Spending Score (1-100)", "Spending Score", "SpendingScore"])

print("Age column:", age_col)
print("Income column:", income_col)
print("Spending Score column:", score_col)

assert all([age_col, income_col, score_col]), "Please set age_col, income_col and score_col manually if names differ."

features = mall[[age_col, income_col, score_col]].copy()
features = features.apply(pd.to_numeric, errors="coerce")
features = features.dropna()

# Scale before K-Means
scaler = StandardScaler()
Z = scaler.fit_transform(features)

# Find a useful K using silhouette score
scores = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(Z)
    scores[k] = silhouette_score(Z, labels)

best_k = max(scores, key=scores.get)
print("Silhouette scores:", scores)
print("Best K by silhouette score:", best_k)


In [ ]:
# The assignment asks to apply K-Means; use the best K from the analysis.
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels = kmeans.fit_predict(Z)

seg = features.copy()
seg["Cluster"] = labels

print(seg.groupby("Cluster").mean(numeric_only=True))
print("\nCluster sizes:")
print(seg["Cluster"].value_counts().sort_index())


In [ ]:
# Visualize Age vs Annual Income
plt.figure(figsize=(8, 6))
plt.scatter(
    seg[age_col],
    seg[income_col],
    c=seg["Cluster"],
    s=70,
    alpha=0.75
)
plt.xlabel("Age")
plt.ylabel("Annual Income")
plt.title("Customer Segmentation — Age vs Income")
plt.show()

# Visualize Annual Income vs Spending Score
plt.figure(figsize=(8, 6))
plt.scatter(
    seg[income_col],
    seg[score_col],
    c=seg["Cluster"],
    s=70,
    alpha=0.75
)
plt.xlabel("Annual Income")
plt.ylabel("Spending Score")
plt.title("Customer Segmentation — Income vs Spending Score")
plt.show()


In [ ]:
# Label clusters as Low / Medium / High Spenders.
# Use the cluster's average spending score for an objective mapping.
cluster_spending = seg.groupby("Cluster")[score_col].mean().sort_values()

n_clusters = len(cluster_spending)

if n_clusters == 3:
    label_map = {
        cluster_spending.index[0]: "Low Spenders",
        cluster_spending.index[1]: "Medium Spenders",
        cluster_spending.index[2]: "High Spenders"
    }
else:
    # General fallback if a different K is selected.
    labels_order = ["Low Spenders", "Medium Spenders", "High Spenders"]
    label_map = {}
    for i, cluster_id in enumerate(cluster_spending.index):
        if i == 0:
            label_map[cluster_id] = labels_order[0]
        elif i == n_clusters - 1:
            label_map[cluster_id] = labels_order[2]
        else:
            label_map[cluster_id] = labels_order[1]

seg["Segment"] = seg["Cluster"].map(label_map)

print("Cluster-to-segment mapping:")
print(label_map)

display(
    seg.groupby("Segment")[[age_col, income_col, score_col]]
       .mean()
       .sort_values(score_col)
)

# Final 2D visualization
plt.figure(figsize=(8, 6))
for segment in seg["Segment"].unique():
    part = seg[seg["Segment"] == segment]
    plt.scatter(
        part[income_col],
        part[score_col],
        s=70,
        alpha=0.75,
        label=segment
    )

plt.xlabel("Annual Income")
plt.ylabel("Spending Score")
plt.title("Customer Segments")
plt.legend()
plt.show()


# Model Saving and Project Documentation

Use joblib to save the fitted model and preprocessing objects. This makes the trained project reusable without retraining every time.


In [ ]:
import joblib

# Example: save Heart Disease Logistic Regression
# joblib.dump(log_reg, "heart_disease_logistic_regression.pkl")
# joblib.dump(scaler, "heart_disease_scaler.pkl")

# Example: save Loan Random Forest + preprocessing
# joblib.dump(rf, "loan_random_forest.pkl")
# joblib.dump(preprocessor, "loan_preprocessor.pkl")

# Example: save Customer Segmentation model + scaler
# joblib.dump(kmeans, "customer_kmeans.pkl")
# joblib.dump(scaler, "customer_scaler.pkl")

print("Uncomment the relevant lines above to save your trained project models.")


# Final Project Checklist

### Option 1 — Heart Disease
- [x] Data cleaning
- [x] Feature scaling
- [x] Logistic Regression
- [x] Decision Tree
- [x] Accuracy / Precision / Recall
- [x] Confusion Matrix
- [x] ROC Curve

### Option 2 — Bank Loan Approval
- [x] Categorical-data preprocessing
- [x] Random Forest
- [x] XGBoost
- [x] Confusion Matrix
- [x] Loan approval prediction

### Option 3 — Customer Segmentation
- [x] K-Means
- [x] Age / Income / Spending Score visualization
- [x] Low / Medium / High Spender labels
- [x] Business-oriented interpretation

**Note:** The uploaded Week 4 sheet names Kaggle datasets but does not include their CSV files. Therefore, the Colab notebook uses `files.upload()` so the exact Kaggle CSV can be uploaded and the cells can be executed on the real data.
